In [ ]:
import os
import json
import csv
import random
import pprint
import sqlite3
import itertools
import gradio as gr
from google import genai
from openai import OpenAI
from google.genai import types
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Cargo price APP

Here is a DB including worldwide airport, IATA Code, Country, City, Airport Type and price/Kg.
DB queries include read by airport, IATA Code, Country, City and return airport information if there are 10 or less results. if there are more results give total information if items, LLM can continue interaction with user to reduce items until origin and destination.  
Because DB returns price/Kg LLM will give result of price.

Test result gave that OPEN API with <font color="blue">**gpt-5-nano**</font> model return best result over Llama using model <font color="blue">**gpt-oss:20b**</font> (it froze my PC momentanely) with responses less detailed, after many test and web investigation conclusion is model training is affecting and locally LLM has many limitation.

<font color="blue">**system_message**</font> has to be tunning for better results

Combine both models in chat consume much time and I am not able to complete if few time, it is complicated and there is betters application or frames existing.


In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start AQ. please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

In [ ]:
ollama_url = "http://localhost:11434/v1"

client = OpenAI()

ollama = OpenAI(api_key="ollama", base_url=ollama_url)

# Create database

It is created by CSV file <font color="blue">**international_airports.csv**</font>

In [ ]:
def generate_price_kg(airport_type: str) -> float:
    base = random.randint(3, 10)
    multipliers = {
        "large_airport":  0.20,
        "medium_airport": 0.35,
    }
    multiplier = multipliers.get(airport_type, 0.50)  # default for small, heliport, etc.
    return round(base * multiplier, 2)

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()

    # Enable foreign key enforcement
    cursor.execute("PRAGMA foreign_keys = ON;")

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS countries (
            id            INTEGER  PRIMARY KEY AUTOINCREMENT,
            IATA_Code     TEXT     UNIQUE NOT NULL
                            CHECK(length(IATA_Code) = 3
                              AND IATA_Code = upper(IATA_Code)),   -- exactly 3 uppercase letters
            Airport_Name  TEXT     NOT NULL,
            City          TEXT     NOT NULL,
            Country       TEXT     NOT NULL,
            Country_Code  TEXT     NOT NULL
                            CHECK(length(Country_Code) = 2
                              AND Country_Code = upper(Country_Code)), -- exactly 2 uppercase letters
            Airport_Type  TEXT     NOT NULL
                            CHECK(Airport_Type IN (
                                'large_airport',
                                'medium_airport',
                                'small_airport',
                                'heliport',
                                'seaplane_base',
                                'balloonport',
                                'closed'
                            )),
            price_kg      REAL     NOT NULL
                            CHECK(price_kg > 0)                    -- must be positive
        )
    """)

    conn.commit()

In [ ]:
def insert_airport(cursor, iata, name, city, country, country_code, airport_type):
    """Helper to insert a row, computing price_kg automatically."""
    price = generate_price_kg(airport_type)
    cursor.execute("""
        INSERT INTO countries
            (IATA_Code, Airport_Name, City, Country, Country_Code, Airport_Type, price_kg)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (iata.upper(), name, city, country, country_code.upper(), airport_type, price))

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;")

    with open('international_airports.csv', 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            counter += 1
            # rows = [(row['Country'], row['Region']) for row in reader]
            insert_airport(cursor,row['IATA_Code'],row['Airport_Name'],row['City'],row['Country'],row['Country_Code'],row['Airport_Type'])

    conn.commit()

# Function for querying DB

In [ ]:
SEARCH_FIELDS = {"IATA_Code", "City", "Country", "Country_Code"}

def query_airports(field: str, value: str) -> str:
    if field not in SEARCH_FIELDS:
        return json.dumps({
            "status": "error",
            "message": f"Invalid field '{field}'. Must be one of: {sorted(SEARCH_FIELDS)}",
            "total": 0,
            "data": []
        })

    # Format value based on field type
    UPPER_FIELDS = {"IATA_Code", "Country_Code"}
    TITLE_FIELDS = {"City", "Country"}

    if field in UPPER_FIELDS:
        formatted_value = value.upper()
    elif field in TITLE_FIELDS:
        formatted_value = value.title()
    else:
        formatted_value = value

    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()

        cursor.execute(f"SELECT * FROM countries WHERE {field} = ?", (formatted_value,))
        result = cursor.fetchall()

    if not result:
        return json.dumps({
            "status": "empty",
            "message": f"No airports found for {field} = '{value}'",
            "total": 0,
            "data": []
        })
    elif len(result) > 10:
        return json.dumps({
            "status": "truncated",
            "message": "Too many results, showing total only.",
            "total": len(result),
            "data": []
        })
    else:
        return json.dumps({
            "status": "ok",
            "total": len(result),
            "data": [dict(row) for row in result]
        })

In [ ]:
def get_airport_by_iata(iata_code: str) -> str:
    return query_airports("IATA_Code", iata_code)

def get_airports_by_city(city: str) -> str:
    return query_airports("City", city.capitalize())

def get_airports_by_country(country: str) -> str:
    return query_airports("Country", country.capitalize())

def get_airports_by_country_code(country_code: str) -> str:
    return query_airports("Country_Code", country_code)

# Tool definition mapping for use in LLM and only use registered functions

In [ ]:
TOOL_REGISTRY = {
    "get_airport_by_iata":        get_airport_by_iata,
    "get_airports_by_city":       get_airports_by_city,
    "get_airports_by_country":    get_airports_by_country,
    "get_airports_by_country_code": get_airports_by_country_code,
}

def run_tool(tool_call) -> str:
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    fn = TOOL_REGISTRY.get(name)
    if not fn:
        return json.dumps({"status": "error", "message": f"Unknown tool: {name}"})

    return fn(**args)  

# Chat for API use

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name in TOOL_REGISTRY:
            result = run_tool(tool_call)
            responses.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })
        else:
            # Graceful fallback for unknown tools
            responses.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps({"status": "error", "message": f"Unknown tool: {tool_call.function.name}"})
            })
    return responses

In [ ]:
system_message = """
You are a helpful assistant for a Cargo Airline called CAIrgo607, interacting with user for getting origin and destination of cargo and provide the price base on origin price and destination price.
Give short, courteous answers. 
Always be accurate. If you don't know the answer, say so.
You MUST always use the provided tools to look up airport data, city, country and price for cargo.
Never answer airport queries from your own knowledge.
"""

In [ ]:
def chat(user_input, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": user_input}]
    response = client.chat.completions.create(model="gpt-4o", messages=messages, tools=AIRPORT_TOOLS, tool_choice="required")

    while response.choices[0].finish_reason=="tool_calls":
        assistant_msg = response.choices[0].message
        responses = handle_tool_calls(assistant_msg)
        messages.append(assistant_msg)
        messages.extend(responses)
        response = client.chat.completions.create(model="gpt-4o", messages=messages, tools=AIRPORT_TOOLS)
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

# Local Ollama with model <font color="blue">**gpt-oss:20b**</font>

In [ ]:
def call_ollama(prompt):
    """Local LLM models calls, contenido is dictionary with instruction for system and user roles"""
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    response = ollama.chat.completions.create(
        model =  "gpt-oss:20b",
        messages = messages
    )
    return response.choices[0].message.content


In [ ]:
def chat(user_input, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": user_input}]
    response = ollama.chat.completions.create(model="gpt-oss:20b", messages=messages, tools=AIRPORT_TOOLS, tool_choice="required")

    while response.choices[0].finish_reason=="tool_calls":
        assistant_msg = response.choices[0].message
        responses = handle_tool_calls(assistant_msg)
        messages.append(assistant_msg)
        messages.extend(responses)
        response = ollama.chat.completions.create(model="gpt-oss:20b", messages=messages, tools=AIRPORT_TOOLS)
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()